In [ ]:
import os
import requests
import json
import numpy as np
from pathlib import Path
from typing import List, Dict

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma


In [ ]:
DATA_DIR = Path("../data")
CHROMA_DIR = DATA_DIR / "chroma_db"
IMAGE_DIR = DATA_DIR / "extracted/images"

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embedding_model
)

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [ ]:
#multi query expansion

def expand_query(query: str) -> List[str] :
    return [
        query,
        f"defination of {query}",
        f"{query} explained",
        f"{query} in the document"
    ]

In [ ]:
def retrieve_context(query: str, k: int = 6):
    expanded_queries = expand_query(query)

    all_docs = []

    for q in expanded_queries:
        results = vectorstore.similarity_search(q, k=k)
        for doc in results:
            all_docs.append(doc)
    
    return all_docs


In [ ]:
#to prevent repeated OCR/text chunks

def deduplicate_chunks(docs, threshold: float = 0.92):
    unique_docs = []
    stored_embeddings = []

    for doc in docs:
        emb = embedding_model.embed_documents([doc.page_content])[0]
        is_duplicate = False
        for stored_emb in stored_embeddings:
            if cosine_similarity(emb, stored_emb) > threshold:
                is_duplicate = True
                break
        
        if not is_duplicate:
            unique_docs.append(doc)
            stored_embeddings.append(emb)
        
    return unique_docs

In [ ]:
#citation grounded generation
#LLM is allowed to generate an answer only using retrieved document chunks,and the answer is explicitly tied back to those chunks (page numbers / context blocks), instead of relying on its own memory.
def build_prompt(query: str, docs: List) -> str:
    context_text = ""

    for i, doc in enumerate(docs):
        meta = doc.metadata or {}
        context_text += (
            f"\n[Context {i+1} | page {meta.get('page','?')}]\n"
            f"{doc.page_content}\n"
        )

    instruction = (
        "Answer ONLY using the given context.\n"
        "If the answer is not present in the context, say:\n"
        "'Not specified in the provided document.'\n"
        "Do NOT guess or add outside knowledge."
    )
    return f"""
You are a technical assistant.

{instruction}

Context:
{context_text}

Question:
{query}

Answer:
"""

In [69]:
OPENROUTER_API_KEY = "ENTER-YOUR-API-KEY"

def call_llm(prompt: str) -> Dict:
    response = requests.post(
        url="https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENROUTER_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": "google/gemma-3-27b-it:free",
            "messages": [
                {"role": "user", "content": prompt}
            ]
        }
    )

    data = response.json()
    text = data["choices"][0]["message"]["content"]

    return {
        "text": text,
        "raw_response": data
    }


In [70]:
def retrieve_images_for_query(query: str):
    #placeholder for future CLIP / vision ranking
    return []

In [72]:
def rag_answer(query: str) -> Dict:
    retrieved_docs = retrieve_context(query, k=6)
    retrieved_docs = deduplicate_chunks(retrieved_docs)

    prompt = build_prompt(query, retrieved_docs)
    llm_result = call_llm(prompt)

    return {
        "query": query,
        "answer": llm_result["text"],
        "citations": [
            {
                "content": doc.page_content,
                "metadata": doc.metadata
            }
            for doc in retrieved_docs
        ],
        "images": retrieve_images_for_query(query)
    }


In [67]:
response = rag_answer("How do linguistic diversity metrics such as Self-BLEU, n-gram entropy, and syntactic parse depth demonstrate that AutoFactory balances semantic consistency with expressive variation?")
response

{'query': 'How do linguistic diversity metrics such as Self-BLEU, n-gram entropy, and syntactic parse depth demonstrate that AutoFactory balances semantic consistency with expressive variation?',
 'answer': 'Self-BLEU scores for AutoFactory are high (average of 0.9984, minimum of 0.9510, standard deviation of 0.0045) reflecting the dataset’s design where specifications reformulate the same base scenario using consistent technical components. Despite this consistency, LLM-generated reformulations still produce varied sentence constructions, preserving diversity in expression. Additionally, the dataset contains 8078 distinct trigrams and 3300 bigrams, with trigram entropy reaching 10.56, indicating a high level of lexical diversity and reduced repetition. High entropy values suggest the dataset avoids rigid or repetitive phrasing, offering a broader lexical space. System 3 exhibits the highest bigram and trigram entropy, consistent with its longer and more structurally varied sentences.\

In [ ]:
response = rag_answer("According to Table 2, how many total tokens are labeled as ACTUATOR across training, validation, and test sets, and what percentage of the dataset do they represent?")
response

In [ ]:
response = rag_answer("What problem does the AutoFactory dataset aim to solve in industrial automation, and why were existing datasets insufficient?")
response


In [ ]:
response = rag_answer("Which five Named Entity Recognition (NER) categories are used in the AutoFactory dataset, and what real-world components do they represent?")

response


In [ ]:
response = rag_answer("When was the AutoFactory article received, revised, accepted, and made available online, and where is the dataset hosted publicly?")
response

In [ ]:
response = rag_answer("Which large language models were used to generate augmented requirement specifications, and how was semantic consistency verified?")
response

In [ ]:
response = rag_answer("According to Table 2, how many total tokens are labeled as ACTUATOR across training, validation, and test sets, and what percentage of the dataset do they represent?")
response

In [ ]:
response = rag_answer("What is the train–validation–test split used in the AutoFactory dataset, and how many specifications are included in each split?")
response

In [ ]:
response = rag_answer("How does AutoFactory differ from CoNLL-2003, FabNER, and MS-NERC in terms of domain focus, annotation purpose, and NER tag design?")
response

In [ ]:
response = rag_answer("Explain how Factory I/O is used in building the dataset and why grounding annotations in simulated industrial systems is important.")
response

In [ ]:
response = rag_answer("Describe the three-phase methodology used to construct the AutoFactory dataset and explain how each phase contributes to dataset reliability.")
response


In [ ]:
response = rag_answer("How do linguistic diversity metrics such as Self-BLEU, n-gram entropy, and syntactic parse depth demonstrate that AutoFactory balances semantic consistency with expressive variation?")
response